In [1]:
import os, random, pandas as pd, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.utils import shuffle

# ===============================
# Constants & Reproducibility
# ===============================
SEED = 1337
IMG_SIZE = (224, 224)
BATCH_SIZE = 8
AUTOTUNE = tf.data.AUTOTUNE
random.seed(SEED); tf.random.set_seed(SEED)

# ===============================
# CSV manifest (fixed split)
# ===============================
# If CSV has absolute paths, leave BASE_DIR=None. If relative, set BASE_DIR.
CSV_PATH = "/home/jovyan/Independent/split_manifest.csv"
BASE_DIR = None  # e.g., "/home/jovyan/.../Breast Cancer" if your CSV paths are relative

manifest = pd.read_csv(CSV_PATH)

# Resolve to absolute paths if needed
if BASE_DIR is not None:
    manifest["path"] = manifest["path"].apply(
        lambda p: p if os.path.isabs(p) else os.path.join(BASE_DIR, p)
    )

# Stable label mapping (replaces class_names)
labels_sorted = sorted(manifest["label"].unique())
label_to_int  = {l:i for i,l in enumerate(labels_sorted)}
int_to_label  = {v:k for k,v in label_to_int.items()}
num_classes   = len(labels_sorted)
print("Classes:", labels_sorted)

# Split dataframes
df_train = manifest[manifest["split"]=="train"].copy()
df_val   = manifest[manifest["split"]=="val"].copy()
df_test  = manifest[manifest["split"]=="test"].copy()

print("Counts by split:\n", manifest.groupby(["split","label"]).size().unstack(fill_value=0))

# ===============================
# tf.data from CSV (non-leaking)
# ===============================
def _decode(path):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE, antialias=True)
    img = tf.cast(img, tf.float32)  # 0..255; we'll rescale in the model
    return img

def make_ds(df, split, batch_size=BATCH_SIZE, seed=SEED):
    if split == "train":
        df = shuffle(df, random_state=seed)
    paths  = df["path"].tolist()
    labels = [label_to_int[l] for l in df["label"]]
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    def _load(p, y):
        return _decode(p), tf.cast(y, tf.int32)
    ds = ds.map(_load, num_parallel_calls=AUTOTUNE)
    if split == "train":
        ds = ds.shuffle(buffer_size=len(paths), seed=seed, reshuffle_each_iteration=True)
    return ds.batch(batch_size).prefetch(AUTOTUNE)

train_ds = make_ds(df_train, "train")
val_ds   = make_ds(df_val,   "val")
test_ds  = make_ds(df_test,  "test")

# ===============================
# Augment & Normalize blocks
# ===============================
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
])

def normalize_block(x):
    # Scales from 0..255 (our decode) to 0..1 for all models
    return layers.Rescaling(1./255)(x)

# ===============================
# Classifier head (shared)
# ===============================
def classifier_head(x, num_classes, dropout=0.3):
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    return layers.Dense(num_classes, activation="softmax")(x)

# ===============================
# 1) Baseline CNN (from scratch)
# ===============================
def build_baseline_cnn(input_shape, num_classes):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)

    # Three conv blocks, two convs per block (as in your first snippet's baseline)
    for filters in [32, 64, 128]:
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(0.25)(x)

    x = layers.Flatten()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="baseline_cnn")
# ===============================
# 2) ResNet50 (transfer learning)
# ===============================
from tensorflow.keras.applications import ResNet50
def build_resnet50(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = ResNet50(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="resnet50")

# ===============================
# 3) VGG16
# ===============================
from tensorflow.keras.applications import VGG16
def build_vgg16(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = VGG16(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="vgg16")

# ===============================
# 4) InceptionV3
# ===============================
from tensorflow.keras.applications import InceptionV3
def build_inceptionv3(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = InceptionV3(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="inceptionv3")

# ===============================
# 5) EfficientNetB0
# ===============================
from tensorflow.keras.applications import EfficientNetB0
def build_efficientnetb0(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = EfficientNetB0(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="efficientnetb0")

# ===============================
# Pick a model to train
# ===============================
MODEL_NAME = "cnn"   # "cnn" | "resnet" | "vgg" | "inception" | "efficientnet"
WEIGHTS = "imagenet"    # "imagenet" or None
FINE_TUNE_BASE = False  # set True to unfreeze backbone later
input_shape = (*IMG_SIZE, 3)

if MODEL_NAME == "cnn":
    model = build_baseline_cnn(input_shape, num_classes)
elif MODEL_NAME == "resnet":
    model = build_resnet50(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "vgg":
    model = build_vgg16(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "inception":
    model = build_inceptionv3(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "efficientnet":
    model = build_efficientnetb0(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
else:
    raise ValueError("Unknown MODEL_NAME")

# =========================================
# Compile & Train
# =========================================
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

callbacks = [
    keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True, monitor="val_accuracy"),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks
)

# ===== Evaluate on the held-out TEST set (matches second snippet practice) =====
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"{model.name}  |  Test acc: {test_acc:.4f}  |  Test loss: {test_loss:.4f}")

2025-10-10 21:56:34.489301: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-10 21:56:35.483866: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


Classes: ['brain_glioma', 'brain_menin', 'brain_tumor']
Counts by split:
 label  brain_glioma  brain_menin  brain_tumor
split                                        
test            750          750          750
train          3499         3499         3499
val             750          750          750


2025-10-10 21:56:38.431332: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1639] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 949 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:41:00.0, compute capability: 8.6
2025-10-10 21:56:38.433545: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1639] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 21964 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:61:00.0, compute capability: 8.6
2025-10-10 21:56:38.435395: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1639] Created device /job:localhost/replica:0/task:0/device:GPU:2 with 21964 MB memory:  -> device: 2, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:81:00.0, compute capability: 8.6


Epoch 1/20


2025-10-10 21:56:40.151647: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:954] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inbaseline_cnn/dropout/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
2025-10-10 21:56:45.506214: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:432] Loaded cuDNN version 8600
2025-10-10 21:56:45.743882: I tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:606] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.
2025-10-10 21:56:45.747252: I tensorflow/compiler/xla/service/service.cc:168] XLA service 0x5602b07f4180 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-10-10 21:56:45.747297: I tensorflow/compiler/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce RTX 3090, Compute Capability 8.6
2025-10-10 21:56:45.747309: I tensorflow/compiler/xla/service/service.cc:176]   Str

1313/1313 [==============================] - ETA: 0s - loss: 0.8227 - accuracy: 0.6081

2025-10-10 21:57:05.927443: W tensorflow/tsl/framework/bfc_allocator.cc:296] Allocator (GPU_0_bfc) ran out of memory trying to allocate 545.33MiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.
2025-10-10 21:57:05.992570: W tensorflow/tsl/framework/bfc_allocator.cc:296] Allocator (GPU_0_bfc) ran out of memory trying to allocate 560.45MiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.


1313/1313 [==============================] - 29s 15ms/step - loss: 0.8227 - accuracy: 0.6081 - val_loss: 0.6667 - val_accuracy: 0.7236
Epoch 2/20
1313/1313 [==============================] - 24s 15ms/step - loss: 0.6420 - accuracy: 0.7300 - val_loss: 0.5884 - val_accuracy: 0.7480
Epoch 3/20
1313/1313 [==============================] - 24s 15ms/step - loss: 0.5714 - accuracy: 0.7656 - val_loss: 0.5387 - val_accuracy: 0.7991
Epoch 4/20
1313/1313 [==============================] - 23s 14ms/step - loss: 0.5348 - accuracy: 0.7822 - val_loss: 0.4578 - val_accuracy: 0.8262
Epoch 5/20
1313/1313 [==============================] - 24s 15ms/step - loss: 0.4947 - accuracy: 0.8022 - val_loss: 0.4254 - val_accuracy: 0.8396
Epoch 6/20
1313/1313 [==============================] - 24s 15ms/step - loss: 0.4650 - accuracy: 0.8132 - val_loss: 0.3933 - val_accuracy: 0.8507
Epoch 7/20
1313/1313 [==============================] - 24s 15ms/step - loss: 0.4144 - accuracy: 0.8381 - val_loss: 0.3351 - val_accura

In [2]:
import os, random, pandas as pd, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.utils import shuffle

# ===============================
# Constants & Reproducibility
# ===============================
SEED = 1337
IMG_SIZE = (224, 224)
BATCH_SIZE = 8
AUTOTUNE = tf.data.AUTOTUNE
random.seed(SEED); tf.random.set_seed(SEED)

# ===============================
# CSV manifest (fixed split)
# ===============================
# If CSV has absolute paths, leave BASE_DIR=None. If relative, set BASE_DIR.
CSV_PATH = "/home/jovyan/Independent/split_manifest.csv"
BASE_DIR = None  # e.g., "/home/jovyan/.../Breast Cancer" if your CSV paths are relative

manifest = pd.read_csv(CSV_PATH)

# Resolve to absolute paths if needed
if BASE_DIR is not None:
    manifest["path"] = manifest["path"].apply(
        lambda p: p if os.path.isabs(p) else os.path.join(BASE_DIR, p)
    )

# Stable label mapping (replaces class_names)
labels_sorted = sorted(manifest["label"].unique())
label_to_int  = {l:i for i,l in enumerate(labels_sorted)}
int_to_label  = {v:k for k,v in label_to_int.items()}
num_classes   = len(labels_sorted)
print("Classes:", labels_sorted)

# Split dataframes
df_train = manifest[manifest["split"]=="train"].copy()
df_val   = manifest[manifest["split"]=="val"].copy()
df_test  = manifest[manifest["split"]=="test"].copy()

print("Counts by split:\n", manifest.groupby(["split","label"]).size().unstack(fill_value=0))

# ===============================
# tf.data from CSV (non-leaking)
# ===============================
def _decode(path):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE, antialias=True)
    img = tf.cast(img, tf.float32)  # 0..255; we'll rescale in the model
    return img

def make_ds(df, split, batch_size=BATCH_SIZE, seed=SEED):
    if split == "train":
        df = shuffle(df, random_state=seed)
    paths  = df["path"].tolist()
    labels = [label_to_int[l] for l in df["label"]]
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    def _load(p, y):
        return _decode(p), tf.cast(y, tf.int32)
    ds = ds.map(_load, num_parallel_calls=AUTOTUNE)
    if split == "train":
        ds = ds.shuffle(buffer_size=len(paths), seed=seed, reshuffle_each_iteration=True)
    return ds.batch(batch_size).prefetch(AUTOTUNE)

train_ds = make_ds(df_train, "train")
val_ds   = make_ds(df_val,   "val")
test_ds  = make_ds(df_test,  "test")

# ===============================
# Augment & Normalize blocks
# ===============================
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
])

def normalize_block(x):
    # Scales from 0..255 (our decode) to 0..1 for all models
    return layers.Rescaling(1./255)(x)

# ===============================
# Classifier head (shared)
# ===============================
def classifier_head(x, num_classes, dropout=0.3):
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    return layers.Dense(num_classes, activation="softmax")(x)

# ===============================
# 1) Baseline CNN (from scratch)
# ===============================
def build_baseline_cnn(input_shape, num_classes):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    for filters in [32, 64, 128]:
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(0.25)(x)
    x = layers.Flatten()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="baseline_cnn")

# ===============================
# 2) ResNet50 (transfer learning)
# ===============================
from tensorflow.keras.applications import ResNet50
def build_resnet50(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = ResNet50(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="resnet50")

# ===============================
# 3) VGG16
# ===============================
from tensorflow.keras.applications import VGG16
def build_vgg16(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = VGG16(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="vgg16")

# ===============================
# 4) InceptionV3
# ===============================
from tensorflow.keras.applications import InceptionV3
def build_inceptionv3(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = InceptionV3(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="inceptionv3")

# ===============================
# 5) EfficientNetB0
# ===============================
from tensorflow.keras.applications import EfficientNetB0
def build_efficientnetb0(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = EfficientNetB0(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="efficientnetb0")

# ===============================
# Pick a model to train
# ===============================
MODEL_NAME = "resnet"   # "cnn" | "resnet" | "vgg" | "inception" | "efficientnet"
WEIGHTS = "imagenet"    # "imagenet" or None
FINE_TUNE_BASE = False  # set True to unfreeze backbone later
input_shape = (*IMG_SIZE, 3)

if MODEL_NAME == "cnn":
    model = build_baseline_cnn(input_shape, num_classes)
elif MODEL_NAME == "resnet":
    model = build_resnet50(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "vgg":
    model = build_vgg16(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "inception":
    model = build_inceptionv3(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "efficientnet":
    model = build_efficientnetb0(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
else:
    raise ValueError("Unknown MODEL_NAME")

# =========================================
# Compile & Train
# =========================================
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

callbacks = [
    keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True, monitor="val_accuracy"),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks
)

# ===== Evaluate on the held-out TEST set (matches second snippet practice) =====
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"{model.name}  |  Test acc: {test_acc:.4f}  |  Test loss: {test_loss:.4f}")

Classes: ['brain_glioma', 'brain_menin', 'brain_tumor']
Counts by split:
 label  brain_glioma  brain_menin  brain_tumor
split                                        
test            750          750          750
train          3499         3499         3499
val             750          750          750
Epoch 1/20
1313/1313 [==============================] - 29s 16ms/step - loss: 1.0455 - accuracy: 0.4560 - val_loss: 0.9085 - val_accuracy: 0.6080
Epoch 2/20
1313/1313 [==============================] - 24s 15ms/step - loss: 0.9319 - accuracy: 0.5633 - val_loss: 0.8467 - val_accuracy: 0.6591
Epoch 3/20
1313/1313 [==============================] - 25s 15ms/step - loss: 0.8997 - accuracy: 0.5784 - val_loss: 0.7942 - val_accuracy: 0.6627
Epoch 4/20
1313/1313 [==============================] - 24s 15ms/step - loss: 0.8720 - accuracy: 0.6001 - val_loss: 0.8150 - val_accuracy: 0.6653
Epoch 5/20
1313/1313 [==============================] - 24s 15ms/step - loss: 0.8539 - accuracy: 0.6090 - val_lo

In [4]:
import os, random, pandas as pd, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.utils import shuffle

# ===============================
# Constants & Reproducibility
# ===============================
SEED = 1337
IMG_SIZE = (224, 224)
BATCH_SIZE = 8
AUTOTUNE = tf.data.AUTOTUNE
random.seed(SEED); tf.random.set_seed(SEED)

# ===============================
# CSV manifest (fixed split)
# ===============================
# If CSV has absolute paths, leave BASE_DIR=None. If relative, set BASE_DIR.
CSV_PATH = "/home/jovyan/Independent/split_manifest.csv"
BASE_DIR = None  # e.g., "/home/jovyan/.../Breast Cancer" if your CSV paths are relative

manifest = pd.read_csv(CSV_PATH)

# Resolve to absolute paths if needed
if BASE_DIR is not None:
    manifest["path"] = manifest["path"].apply(
        lambda p: p if os.path.isabs(p) else os.path.join(BASE_DIR, p)
    )

# Stable label mapping (replaces class_names)
labels_sorted = sorted(manifest["label"].unique())
label_to_int  = {l:i for i,l in enumerate(labels_sorted)}
int_to_label  = {v:k for k,v in label_to_int.items()}
num_classes   = len(labels_sorted)
print("Classes:", labels_sorted)

# Split dataframes
df_train = manifest[manifest["split"]=="train"].copy()
df_val   = manifest[manifest["split"]=="val"].copy()
df_test  = manifest[manifest["split"]=="test"].copy()

print("Counts by split:\n", manifest.groupby(["split","label"]).size().unstack(fill_value=0))

# ===============================
# tf.data from CSV (non-leaking)
# ===============================
def _decode(path):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE, antialias=True)
    img = tf.cast(img, tf.float32)  # 0..255; we'll rescale in the model
    return img

def make_ds(df, split, batch_size=BATCH_SIZE, seed=SEED):
    if split == "train":
        df = shuffle(df, random_state=seed)
    paths  = df["path"].tolist()
    labels = [label_to_int[l] for l in df["label"]]
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    def _load(p, y):
        return _decode(p), tf.cast(y, tf.int32)
    ds = ds.map(_load, num_parallel_calls=AUTOTUNE)
    if split == "train":
        ds = ds.shuffle(buffer_size=len(paths), seed=seed, reshuffle_each_iteration=True)
    return ds.batch(batch_size).prefetch(AUTOTUNE)

train_ds = make_ds(df_train, "train")
val_ds   = make_ds(df_val,   "val")
test_ds  = make_ds(df_test,  "test")

# ===============================
# Augment & Normalize blocks
# ===============================
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
])

def normalize_block(x):
    # Scales from 0..255 (our decode) to 0..1 for all models
    return layers.Rescaling(1./255)(x)

# ===============================
# Classifier head (shared)
# ===============================
def classifier_head(x, num_classes, dropout=0.3):
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    return layers.Dense(num_classes, activation="softmax")(x)

# ===============================
# 1) Baseline CNN (from scratch)
# ===============================
def build_baseline_cnn(input_shape, num_classes):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)

    # Three conv blocks, two convs per block (as in your first snippet's baseline)
    for filters in [32, 64, 128]:
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(0.25)(x)

    x = layers.Flatten()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="baseline_cnn")
# ===============================
# 2) ResNet50 (transfer learning)
# ===============================
from tensorflow.keras.applications import ResNet50
def build_resnet50(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = ResNet50(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="resnet50")

# ===============================
# 3) VGG16
# ===============================
from tensorflow.keras.applications import VGG16
def build_vgg16(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = VGG16(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="vgg16")

# ===============================
# 4) InceptionV3
# ===============================
from tensorflow.keras.applications import InceptionV3
def build_inceptionv3(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = InceptionV3(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="inceptionv3")

# ===============================
# 5) EfficientNetB0
# ===============================
from tensorflow.keras.applications import EfficientNetB0
def build_efficientnetb0(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = EfficientNetB0(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="efficientnetb0")

# ===============================
# Pick a model to train
# ===============================
MODEL_NAME = "vgg"   # "cnn" | "resnet" | "vgg" | "inception" | "efficientnet"
WEIGHTS = "imagenet"    # "imagenet" or None
FINE_TUNE_BASE = False  # set True to unfreeze backbone later
input_shape = (*IMG_SIZE, 3)

if MODEL_NAME == "cnn":
    model = build_baseline_cnn(input_shape, num_classes)
elif MODEL_NAME == "resnet":
    model = build_resnet50(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "vgg":
    model = build_vgg16(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "inception":
    model = build_inceptionv3(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "efficientnet":
    model = build_efficientnetb0(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
else:
    raise ValueError("Unknown MODEL_NAME")

# =========================================
# Compile & Train
# =========================================
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

callbacks = [
    keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True, monitor="val_accuracy"),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks
)

# ===== Evaluate on the held-out TEST set (matches second snippet practice) =====
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"{model.name}  |  Test acc: {test_acc:.4f}  |  Test loss: {test_loss:.4f}")

Classes: ['brain_glioma', 'brain_menin', 'brain_tumor']
Counts by split:
 label  brain_glioma  brain_menin  brain_tumor
split                                        
test            750          750          750
train          3499         3499         3499
val             750          750          750
Epoch 1/20
1313/1313 [==============================] - 30s 19ms/step - loss: 0.7252 - accuracy: 0.7108 - val_loss: 0.5392 - val_accuracy: 0.8169
Epoch 2/20
1313/1313 [==============================] - 28s 18ms/step - loss: 0.5239 - accuracy: 0.8040 - val_loss: 0.4370 - val_accuracy: 0.8511
Epoch 3/20
1313/1313 [==============================] - 28s 18ms/step - loss: 0.4684 - accuracy: 0.8189 - val_loss: 0.3816 - val_accuracy: 0.8738
Epoch 4/20
1313/1313 [==============================] - 28s 19ms/step - loss: 0.4392 - accuracy: 0.8310 - val_loss: 0.3483 - val_accuracy: 0.8871
Epoch 5/20
1313/1313 [==============================] - 28s 19ms/step - loss: 0.4234 - accuracy: 0.8375 - val_lo

In [5]:
#GoogLeNet
import os, random, pandas as pd, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.utils import shuffle

# ===============================
# Constants & Reproducibility
# ===============================
SEED = 1337
IMG_SIZE = (224, 224)
BATCH_SIZE = 8
AUTOTUNE = tf.data.AUTOTUNE
random.seed(SEED); tf.random.set_seed(SEED)

# ===============================
# CSV manifest (fixed split)
# ===============================
# If CSV has absolute paths, leave BASE_DIR=None. If relative, set BASE_DIR.
CSV_PATH = "/home/jovyan/Independent/split_manifest.csv"
BASE_DIR = None  # e.g., "/home/jovyan/.../Breast Cancer" if your CSV paths are relative

manifest = pd.read_csv(CSV_PATH)

# Resolve to absolute paths if needed
if BASE_DIR is not None:
    manifest["path"] = manifest["path"].apply(
        lambda p: p if os.path.isabs(p) else os.path.join(BASE_DIR, p)
    )

# Stable label mapping (replaces class_names)
labels_sorted = sorted(manifest["label"].unique())
label_to_int  = {l:i for i,l in enumerate(labels_sorted)}
int_to_label  = {v:k for k,v in label_to_int.items()}
num_classes   = len(labels_sorted)
print("Classes:", labels_sorted)

# Split dataframes
df_train = manifest[manifest["split"]=="train"].copy()
df_val   = manifest[manifest["split"]=="val"].copy()
df_test  = manifest[manifest["split"]=="test"].copy()

print("Counts by split:\n", manifest.groupby(["split","label"]).size().unstack(fill_value=0))

# ===============================
# tf.data from CSV (non-leaking)
# ===============================
def _decode(path):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE, antialias=True)
    img = tf.cast(img, tf.float32)  # 0..255; we'll rescale in the model
    return img

def make_ds(df, split, batch_size=BATCH_SIZE, seed=SEED):
    if split == "train":
        df = shuffle(df, random_state=seed)
    paths  = df["path"].tolist()
    labels = [label_to_int[l] for l in df["label"]]
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    def _load(p, y):
        return _decode(p), tf.cast(y, tf.int32)
    ds = ds.map(_load, num_parallel_calls=AUTOTUNE)
    if split == "train":
        ds = ds.shuffle(buffer_size=len(paths), seed=seed, reshuffle_each_iteration=True)
    return ds.batch(batch_size).prefetch(AUTOTUNE)

train_ds = make_ds(df_train, "train")
val_ds   = make_ds(df_val,   "val")
test_ds  = make_ds(df_test,  "test")

# ===============================
# Augment & Normalize blocks
# ===============================
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
])

def normalize_block(x):
    # Scales from 0..255 (our decode) to 0..1 for all models
    return layers.Rescaling(1./255)(x)

# ===============================
# Classifier head (shared)
# ===============================
def classifier_head(x, num_classes, dropout=0.3):
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    return layers.Dense(num_classes, activation="softmax")(x)

# ===============================
# 1) Baseline CNN (from scratch)
# ===============================
def build_baseline_cnn(input_shape, num_classes):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)

    # Three conv blocks, two convs per block (as in your first snippet's baseline)
    for filters in [32, 64, 128]:
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(0.25)(x)

    x = layers.Flatten()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="baseline_cnn")
# ===============================
# 2) ResNet50 (transfer learning)
# ===============================
from tensorflow.keras.applications import ResNet50
def build_resnet50(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = ResNet50(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="resnet50")

# ===============================
# 3) VGG16
# ===============================
from tensorflow.keras.applications import VGG16
def build_vgg16(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = VGG16(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="vgg16")

# ===============================
# 4) InceptionV3
# ===============================
from tensorflow.keras.applications import InceptionV3
def build_inceptionv3(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = InceptionV3(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="inceptionv3")

# ===============================
# 5) EfficientNetB0
# ===============================
from tensorflow.keras.applications import EfficientNetB0
def build_efficientnetb0(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = EfficientNetB0(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="efficientnetb0")

# ===============================
# Pick a model to train
# ===============================
MODEL_NAME = "inception"   # "cnn" | "resnet" | "vgg" | "inception" | "efficientnet"
WEIGHTS = "imagenet"    # "imagenet" or None
FINE_TUNE_BASE = False  # set True to unfreeze backbone later
input_shape = (*IMG_SIZE, 3)

if MODEL_NAME == "cnn":
    model = build_baseline_cnn(input_shape, num_classes)
elif MODEL_NAME == "resnet":
    model = build_resnet50(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "vgg":
    model = build_vgg16(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "inception":
    model = build_inceptionv3(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "efficientnet":
    model = build_efficientnetb0(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
else:
    raise ValueError("Unknown MODEL_NAME")

# =========================================
# Compile & Train
# =========================================
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

callbacks = [
    keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True, monitor="val_accuracy"),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks
)

# ===== Evaluate on the held-out TEST set (matches second snippet practice) =====
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"{model.name}  |  Test acc: {test_acc:.4f}  |  Test loss: {test_loss:.4f}")

Classes: ['brain_glioma', 'brain_menin', 'brain_tumor']
Counts by split:
 label  brain_glioma  brain_menin  brain_tumor
split                                        
test            750          750          750
train          3499         3499         3499
val             750          750          750
Epoch 1/20
1313/1313 [==============================] - 28s 15ms/step - loss: 0.4395 - accuracy: 0.8203 - val_loss: 0.5462 - val_accuracy: 0.8018
Epoch 2/20
1313/1313 [==============================] - 22s 13ms/step - loss: 0.3372 - accuracy: 0.8705 - val_loss: 0.6223 - val_accuracy: 0.7920
Epoch 3/20
1313/1313 [==============================] - 21s 12ms/step - loss: 0.3225 - accuracy: 0.8765 - val_loss: 0.2322 - val_accuracy: 0.9133
Epoch 4/20
1313/1313 [==============================] - 21s 13ms/step - loss: 0.3208 - accuracy: 0.8779 - val_loss: 0.2829 - val_accuracy: 0.9013
Epoch 5/20
1313/1313 [==============================] - 21s 13ms/step - loss: 0.2962 - accuracy: 0.8896 - val_lo

In [7]:
#Effnecitnet 
import os, random, pandas as pd, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.utils import shuffle

# ===============================
# Constants & Reproducibility
# ===============================
SEED = 1337
IMG_SIZE = (224, 224)
BATCH_SIZE = 8
AUTOTUNE = tf.data.AUTOTUNE
random.seed(SEED); tf.random.set_seed(SEED)

# ===============================
# CSV manifest (fixed split)
# ===============================
# If CSV has absolute paths, leave BASE_DIR=None. If relative, set BASE_DIR.
CSV_PATH = "/home/jovyan/Independent/split_manifest.csv"
BASE_DIR = None  # e.g., "/home/jovyan/.../Breast Cancer" if your CSV paths are relative

manifest = pd.read_csv(CSV_PATH)

# Resolve to absolute paths if needed
if BASE_DIR is not None:
    manifest["path"] = manifest["path"].apply(
        lambda p: p if os.path.isabs(p) else os.path.join(BASE_DIR, p)
    )

# Stable label mapping (replaces class_names)
labels_sorted = sorted(manifest["label"].unique())
label_to_int  = {l:i for i,l in enumerate(labels_sorted)}
int_to_label  = {v:k for k,v in label_to_int.items()}
num_classes   = len(labels_sorted)
print("Classes:", labels_sorted)

# Split dataframes
df_train = manifest[manifest["split"]=="train"].copy()
df_val   = manifest[manifest["split"]=="val"].copy()
df_test  = manifest[manifest["split"]=="test"].copy()

print("Counts by split:\n", manifest.groupby(["split","label"]).size().unstack(fill_value=0))

# ===============================
# tf.data from CSV (non-leaking)
# ===============================
def _decode(path):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE, antialias=True)
    img = tf.cast(img, tf.float32)  # 0..255; we'll rescale in the model
    return img

def make_ds(df, split, batch_size=BATCH_SIZE, seed=SEED):
    if split == "train":
        df = shuffle(df, random_state=seed)
    paths  = df["path"].tolist()
    labels = [label_to_int[l] for l in df["label"]]
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    def _load(p, y):
        return _decode(p), tf.cast(y, tf.int32)
    ds = ds.map(_load, num_parallel_calls=AUTOTUNE)
    if split == "train":
        ds = ds.shuffle(buffer_size=len(paths), seed=seed, reshuffle_each_iteration=True)
    return ds.batch(batch_size).prefetch(AUTOTUNE)

train_ds = make_ds(df_train, "train")
val_ds   = make_ds(df_val,   "val")
test_ds  = make_ds(df_test,  "test")

# ===============================
# Augment & Normalize blocks
# ===============================
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
])

def normalize_block(x):
    # Scales from 0..255 (our decode) to 0..1 for all models
    return layers.Rescaling(1./255)(x)

# ===============================
# Classifier head (shared)
# ===============================
def classifier_head(x, num_classes, dropout=0.3):
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    return layers.Dense(num_classes, activation="softmax")(x)

# ===============================
# 1) Baseline CNN (from scratch)
# ===============================
def build_baseline_cnn(input_shape, num_classes):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)

    # Three conv blocks, two convs per block (as in your first snippet's baseline)
    for filters in [32, 64, 128]:
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(0.25)(x)

    x = layers.Flatten()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="baseline_cnn")
# ===============================
# 2) ResNet50 (transfer learning)
# ===============================
from tensorflow.keras.applications import ResNet50
def build_resnet50(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = ResNet50(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="resnet50")

# ===============================
# 3) VGG16
# ===============================
from tensorflow.keras.applications import VGG16
def build_vgg16(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = VGG16(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="vgg16")

# ===============================
# 4) InceptionV3
# ===============================
from tensorflow.keras.applications import InceptionV3
def build_inceptionv3(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = InceptionV3(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="inceptionv3")

# ===============================
# 5) EfficientNetB0
# ===============================
from tensorflow.keras.applications import EfficientNetB0
def build_efficientnetb0(input_shape, num_classes, weights="imagenet", train_base=False):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalize_block(x)
    base = EfficientNetB0(include_top=False, weights=weights, input_tensor=x)
    base.trainable = train_base
    outputs = classifier_head(base.output, num_classes, dropout=0.3)
    return keras.Model(inputs, outputs, name="efficientnetb0")

# ===============================
# Pick a model to train
# ===============================
MODEL_NAME = "efficientnet"   # "cnn" | "resnet" | "vgg" | "inception" | "efficientnet"
WEIGHTS = "imagenet"    # "imagenet" or None
FINE_TUNE_BASE = False  # set True to unfreeze backbone later
input_shape = (*IMG_SIZE, 3)

if MODEL_NAME == "cnn":
    model = build_baseline_cnn(input_shape, num_classes)
elif MODEL_NAME == "resnet":
    model = build_resnet50(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "vgg":
    model = build_vgg16(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "inception":
    model = build_inceptionv3(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
elif MODEL_NAME == "efficientnet":
    model = build_efficientnetb0(input_shape, num_classes, weights=WEIGHTS, train_base=FINE_TUNE_BASE)
else:
    raise ValueError("Unknown MODEL_NAME")

# =========================================
# Compile & Train
# =========================================
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

callbacks = [
    keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True, monitor="val_accuracy"),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks
)

# ===== Evaluate on the held-out TEST set (matches second snippet practice) =====
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"{model.name}  |  Test acc: {test_acc:.4f}  |  Test loss: {test_loss:.4f}")

Classes: ['brain_glioma', 'brain_menin', 'brain_tumor']
Counts by split:
 label  brain_glioma  brain_menin  brain_tumor
split                                        
test            750          750          750
train          3499         3499         3499
val             750          750          750
Epoch 1/20
1313/1313 [==============================] - 26s 13ms/step - loss: 1.1319 - accuracy: 0.3297 - val_loss: 1.1228 - val_accuracy: 0.3333
Epoch 2/20
1313/1313 [==============================] - 21s 12ms/step - loss: 1.1251 - accuracy: 0.3343 - val_loss: 1.1009 - val_accuracy: 0.3333
Epoch 3/20
1313/1313 [==============================] - 20s 12ms/step - loss: 1.1296 - accuracy: 0.3327 - val_loss: 1.1145 - val_accuracy: 0.3333
Epoch 4/20
1313/1313 [==============================] - 20s 12ms/step - loss: 1.1241 - accuracy: 0.3385 - val_loss: 1.1072 - val_accuracy: 0.3333
Epoch 5/20
1313/1313 [==============================] - 20s 12ms/step - loss: 1.1246 - accuracy: 0.3287 - val_lo